In [ ]:
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

import scipy
from scipy import stats
import math

import numpy as np

# Основной пайплайн Sklavenitis

В этом ноутбуке собраны основные шаги подготовки данных для будущего матчинга товаров между магазинами.

In [ ]:
base_path = Path(r".\raw_data")

sklavenitis = pd.read_csv(base_path / "sklavenitis-ypsonas.csv", encoding="utf-8")
alphamega = pd.read_csv(base_path / "alphamega.csv", encoding="utf-8")
alphamega
# list(alphamega.brand.drop_duplicates().sort_values())

In [ ]:
alphamega

In [ ]:
sklavenitis

In [ ]:
sklavenitis.promo_label.unique()

In [ ]:
sklavenitis.promo_label.notna().sum()

In [ ]:
sklavenitis.category_path.unique()

In [ ]:
sklavenitis.query("brand == 'Barilla'")

In [ ]:
sklavenitis.brand.unique()

In [ ]:
# list(sklavenitis.brand.drop_duplicates().sort_values())

In [ ]:
# sklavenitis[['brand_clean', 'product_type']] = sklavenitis['brand'].str.split(' ', n=1, expand=True)

In [ ]:
# sklavenitis

In [ ]:
sklavenitis = sklavenitis.drop(columns=['image_url', 'product_url', 'sku'])

In [ ]:
sklavenitis= sklavenitis.drop(columns=['availability'])

In [ ]:
# sklavenitis['brand_clean'] = sklavenitis['brand_clean'].str.capitalize()
# sklavenitis['product_type'] = sklavenitis['product_type'].str.capitalize()
# sklavenitis

# чтобы удалить строки, где в конкретных столбцах есть пропуски:
# sklavenitis = sklavenitis.dropna(subset=['product_name', 'brand', 'price_unit'])
sklavenitis
sklavenitis = sklavenitis.dropna(subset=['product_name', 'brand'], how='all')
sklavenitis = sklavenitis.dropna(subset=['price', 'price_unit'], how='all')
sklavenitis.price.count()

In [ ]:
# sklavenitis

## 1. Очистка и нормализация названия товара

Здесь мы приводим product_name к более чистому и единообразному виду, чтобы потом проще было извлекать размеры, grade и собирать product_core.

In [ ]:
# нормализуем название продукта:
# приведем все к строчным буквам, 
# уберем пробелы перед началом и после конца,
# заменим все повторяющиеся пробелы, переносы строк, табы на один обычный пробел \s - это все виды пробелов, + - больше одного подряд 
# NEW: убираем невидимые zero-width символы, которые ломают слова вроде b\u200bread

sklavenitis['product_name_clean'] = (
    sklavenitis['product_name']
    .str.lower()
    .str.replace(r'[\u200b\u200c\u200d\ufeff]', '', regex=True)
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)

In [ ]:
sklavenitis

In [ ]:
# приводим к одному виду умножения и запятые с точками
sklavenitis['product_name_clean'] = (
    sklavenitis['product_name_clean']
    .fillna('')
    .str.replace('×', 'x', regex=False)
    .str.replace(',', '.', regex=False)
)

In [ ]:
sklavenitis[sklavenitis['product_name'].str.startswith('KL', na=False)].sort_values(by='product_name')
# sklavenitis

In [ ]:
# удаляем мусорные цифры после единиц измерения в названии

# только для обычных названий заканчивающихся на типа 250g
# sklavenitis['product_name_clean'] = (
#     sklavenitis['product_name_clean']
#     .str.replace(
#         r'(\b\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs))\s+\d+\s*$',
#         r'\1',
#         regex=True
#     )
# )

# для обычных названий и названий типа 9x250ml
sklavenitis['product_name_clean'] = (
    sklavenitis['product_name_clean']
    .str.replace(
        r'(\b(?:\d+\s*[x×]\s*)?\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs))\s+\d+\s*$',
        r'\1',
        regex=True
    )
)

In [ ]:
# чтобы пандас все показывал, а не ...
pd.set_option('display.max_colwidth', None)
sklavenitis

In [ ]:
# посмотреть строки, которые НЕ заканчиваются на единицы измерения

# только для обычных названий заканчивающихся на типа 250g
# sklavenitis[
#     ~sklavenitis['product_name_clean'].str.contains(r'\b\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs)\s*$', regex=True, na=False)
# ][['product_name_clean']].head(30)

# для обычных названий и названий типа 9x250ml
sklavenitis[
    ~sklavenitis['product_name_clean'].str.contains(
        r'\b(?:\d+\s*[x×]\s*)?\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs)\s*$',
        regex=True,
        na=False
    )
][['product_name_clean']].head(40)

In [ ]:
# убрать этот хвост кода в названии

# \s — перед кодом должен быть пробел
# \d{4,} — потом 4 или больше цифр подряд
# \s*$ — и это всё должно быть в конце строки ($)
sklavenitis['product_name_clean'] = (
    sklavenitis['product_name_clean']
    .str.replace(r'\s\d{4,}\s*$', '', regex=True)
)

In [ ]:
# посмотреть снова строки, которые НЕ заканчиваются на единицы измерения
sklavenitis[
    ~sklavenitis['product_name_clean'].str.contains(
        r'\b(?:\d+\s*[x×]\s*)?\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs)\s*$',
        regex=True,
        na=False
    )
][['product_name_clean']].head(40)

In [ ]:
# посмотреть те, что заканчиваются на цифру, но не на нормальный размер
sklavenitis[
    sklavenitis['product_name_clean'].str.contains(r'\d\s*$', regex=True, na=False) &
    ~sklavenitis['product_name_clean'].str.contains(
        r'\b(?:\d+\s*[x×]\s*)?\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs)\s*$',
        regex=True,
        na=False
    )
][['product_name_clean']]

In [ ]:
repr(sklavenitis.loc[570, 'product_name_clean'])

In [ ]:
# убрать этот хвост кода в названии ЕЩЕ РАЗ

# \s — перед кодом должен быть пробел
# \d{4,} — потом 4 или больше цифр подряд
# \s*$ — и это всё должно быть в конце строки ($)
sklavenitis['product_name_clean'] = (
    sklavenitis['product_name_clean']
    .str.replace(r'\s\d{4,}\s*$', '', regex=True)
)

In [ ]:
# повторно посмотреть те, что заканчиваются на цифру, но не на нормальный размер и не на х и число и pack of и число
sklavenitis[
    sklavenitis['product_name_clean'].str.contains(r'\d\s*$', regex=True, na=False) &
    ~sklavenitis['product_name_clean'].str.contains(
        r'(?:'
        r'\b(?:\d+\s*[x×]\s*)?\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs)\s*$'
        r'|'
        r'\b[x×]\s*\d+\s*$'
        r'|'
        r'\bpack\s+of\s+\d+\s*$'
        r'|'
        r'\bno\.?\s+\d+\s*$'
        r'|'
        r'\b(?:\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs))\s*[x×]\s*\d+\s*$'
        r')',
        regex=True,
        na=False
    )
][['product_name_clean']]

In [ ]:
# чтобы убрать хвост для строки 2547, потому что она выглядит мутно в отличие от остальных
sklavenitis['product_name_clean'] = (
    sklavenitis['product_name_clean']
    .str.replace(r'\.\s*162\s*$', '', regex=True)
)

In [ ]:
# опять схлопываем пробелы в конце, если они образовались 
sklavenitis['product_name_clean'] = (
    sklavenitis['product_name_clean']
    .str.strip()
)

In [ ]:
# посмотреть, на что заканчиваются строки 
sklavenitis['product_name_clean'].str[-1].value_counts()

In [ ]:
# посмотреть все странные варианты с символами
sklavenitis[
    sklavenitis['product_name_clean'].str.contains(r'[%\.\-\)\(]\s*$', regex=True, na=False)
][['product_name_clean']]

In [ ]:
# точечно убираем открывающуюся скобку 
sklavenitis['product_name_clean'] = (
    sklavenitis['product_name_clean']
    .str.replace(r'\(\s*$', '', regex=True)
    .str.strip()
)

In [ ]:
# снова посмотреть все странные варианты с символами
sklavenitis[
    sklavenitis['product_name_clean'].str.contains(r'[%\.\-\)\(]\s*$', regex=True, na=False)
][['product_name_clean']]

In [ ]:
# посмотреть без скобок
sklavenitis[
    sklavenitis['product_name_clean'].str.contains(r'[%\.\-]\s*$', regex=True, na=False)
][['product_name_clean']]

In [ ]:
# убрать точку и тире в конце одиночные
sklavenitis['product_name_clean'] = (
    sklavenitis['product_name_clean']
    .str.replace(r'[.\-]\s*$', '', regex=True)
    .str.strip()
)

In [ ]:
# снова посмотреть без скобок
sklavenitis[
    sklavenitis['product_name_clean'].str.contains(r'[%\.\-]\s*$', regex=True, na=False)
][['product_name_clean']]

In [ ]:
# убираем -х%, чтобы не мешалась 
sklavenitis['product_name_clean'] = (
    sklavenitis['product_name_clean']
    .str.replace(r'\s*-\d+%\s*$', '', regex=True)
    .str.strip()
)

In [ ]:
# снова посмотреть варианты с символами на конце
sklavenitis[
    sklavenitis['product_name_clean'].str.contains(r'[%\.\-]\s*$', regex=True, na=False)
][['product_name_clean']]

In [ ]:
# ищем все варианты с % в составе 
# regex=False, потому что % тут мы ищем как обычный символ, не как шаблон
sklavenitis[
    sklavenitis['product_name_clean'].str.contains('%', regex=False, na=False)
][['product_name_clean']]

In [ ]:
# убираем в итоге варианты с +% на конце тоже 
sklavenitis['product_name_clean'] = (
    sklavenitis['product_name_clean']
    .str.replace(r'\s*[+-]\d+%\s*$', '', regex=True)
    .str.strip()
)

## 2. Извлечение size, pack_count и объёма

В этом блоке регулярками достаём вес, объём, количество упаковок и базовый нормализованный размер.

In [ ]:
# достаем вес и количество упаковок
# учитывает такие типы записей:
# Вес / объём одной единицы
# 250g
# 300 g
# 270ml
# 1.2 l
# 1lt
# 1ltr
# 2.2kg
# 750+250ml     NEW
# 85gr     NEW

# Мультиупаковка
# 4x125g
# 4 x 125 g
# 9x250ml
# 3x200g
# 4*250ml     NEW
# 330ml x 6
# 3 sachets x 8g     NEW
# 90g x 3pcs      NEW

# Количество упаковок без веса
# x6
# x 6
# x15pcs     NEW
# pack of 2

# Штуки
# 4pcs
# 4 pcs
# 3 x 54 pcs

# БЫЛО БЕЗ НОВЫХ
# ______________________________________________________________________________________________________________
# # 1. Паттерны
# multipack_before_pattern = r'(\d+)\s*x\s*(\d+(?:\.\d+)?)\s*(g|gr|kg|ml|l|lt|ltr|pcs)\b'
# multipack_after_pattern = r'(\d+(?:\.\d+)?)\s*(g|gr|kg|ml|l|lt|ltr|pcs)\s*x\s*(\d+)\b'
# single_pattern = r'(\d+(?:\.\d+)?)\s*(g|gr|kg|ml|l|lt|ltr|pcs)\b'
# pack_only_x_pattern = r'\bx\s*(\d+)\b'
# pack_of_pattern = r'\bpack\s+of\s+(\d+)\b'

# # 2. Ищем разные форматы
# multi_before = sklavenitis['product_name_clean'].str.extract(multipack_before_pattern)
# multi_after = sklavenitis['product_name_clean'].str.extract(multipack_after_pattern)
# single = sklavenitis['product_name_clean'].str.extract(single_pattern)
# pack_only_x = sklavenitis['product_name_clean'].str.extract(pack_only_x_pattern)
# pack_of = sklavenitis['product_name_clean'].str.extract(pack_of_pattern)

# # 3. Размер как сырой текст
# # np.select как if elif else
# # np.select(
# #     [условие1, условие2, условие3],
# #     [значение1, значение2, значение3],
# #     default=что_поставить_если_ничего_не_подошло
# # )
# sklavenitis['size_raw'] = np.select(
#     [
#         multi_before[0].notna(),
#         multi_after[0].notna(),
#         single[0].notna()
#     ],
#     [
#         sklavenitis['product_name_clean'].str.extract(r'((?:\d+\s*x\s*)\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs)\b)')[0],
#         sklavenitis['product_name_clean'].str.extract(r'(\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs)\s*x\s*\d+\b)')[0],
#         sklavenitis['product_name_clean'].str.extract(r'(\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs)\b)')[0]
#     ],
#     default=np.nan
# )

# # 4. Количество упаковок
# sklavenitis['pack_count'] = np.select(
#     [
#         multi_before[0].notna(),
#         multi_after[0].notna(),
#         pack_only_x[0].notna(),
#         pack_of[0].notna()
#     ],
#     [
#         multi_before[0],
#         multi_after[2],
#         pack_only_x[0],
#         pack_of[0]
#     ],
#     default=1
# )

# # 5. Размер одной единицы
# sklavenitis['unit_size_value'] = np.select(
#     [
#         multi_before[1].notna(),
#         multi_after[0].notna(),
#         single[0].notna()
#     ],
#     [
#         multi_before[1],
#         multi_after[0],
#         single[0]
#     ],
#     default=np.nan
# )

# # единицы измерения
# sklavenitis['unit_size_unit'] = np.select(
#     [
#         multi_before[2].notna(),
#         multi_after[1].notna(),
#         single[1].notna()
#     ],
#     [
#         multi_before[2],
#         multi_after[1],
#         single[1]
#     ],
#     default=np.nan
# )

# # 6. Переводим в числа
# sklavenitis['pack_count'] = pd.to_numeric(sklavenitis['pack_count'], errors='coerce')
# sklavenitis['unit_size_value'] = pd.to_numeric(sklavenitis['unit_size_value'], errors='coerce')

# # 7. Общий размер до нормализации
# sklavenitis['size_total_raw'] = sklavenitis['pack_count'] * sklavenitis['unit_size_value']

# # 8. Нормализация в g / ml / pcs
# def normalize_size(row):
#     value = row['size_total_raw']
#     unit = row['unit_size_unit']
    
#     if pd.isna(value) or pd.isna(unit):
#         return pd.Series([np.nan, np.nan])
    
#     if unit == 'g':
#         return pd.Series([value, 'g'])
#     elif unit == 'kg':
#         return pd.Series([value * 1000, 'g'])
#     elif unit == 'ml':
#         return pd.Series([value, 'ml'])
#     elif unit in ['l', 'lt', 'ltr']:
#         return pd.Series([value * 1000, 'ml'])
#     elif unit == 'pcs':
#         return pd.Series([value, 'pcs'])
#     else:
#         return pd.Series([np.nan, np.nan])

# sklavenitis[['size_total_base', 'size_base_unit']] = sklavenitis.apply(normalize_size, axis=1)

# # 9. Красивый nullable int для количества упаковок
# # делает колонку pack_count целочисленной, но так, чтобы она умела хранить пропуски
# sklavenitis['pack_count'] = sklavenitis['pack_count'].astype('Int64')

# СТАЛО С НОВЫМИ
# ______________________________________________________________________________________________________________
# достаем вес и количество упаковок

# 1. Паттерны
# NEW: здесь уже учитываем короткую запись граммов через gr, например 85gr или 85 gr
multipack_before_pattern = r'(\d+)\s*(?:pcs|sachet(?:s)?)?\s*[x\u00d7*]\s*(\d+(?:\.\d+)?)\s*(g|gr|kg|ml|l|lt|ltr|pcs)\b'




# CHANGED: добавили optional pcs после количества упаковок
# было: ...\s*x\s*(\d+)\b
# стало: ...\s*x\s*(\d+)\s*(?:pcs)?\b
multipack_after_pattern = r'(\d+(?:\.\d+)?)\s*(g|gr|kg|ml|l|lt|ltr|pcs)\s*[x\u00d7*]\s*(\d+)\s*(?:pcs)?\b'

single_pattern = r'(?<!\d[-–—])(?<!\d [-–—])(?<!\d[-–—] )(?<!\d [-–—] )\b(\d+(?:\.\d+)?)\s*(g|gr|kg|ml|l|lt|ltr|pcs)\b'
pack_only_x_pattern = r'\b[x\u00d7]\s*(\d+)\s*(?:pcs)?\b'
pack_of_pattern = r'\bpack\s+of\s+(\d+)\b'

additive_pattern = r'(\d+(?:\.\d+)?)\s*\+\s*(\d+(?:\.\d+)?)\s*(g|gr|kg|ml|l|lt|ltr)\b'

# 2. Ищем разные форматы
# NEW: single_pattern теперь тоже ловит короткую запись граммов через gr, например 85gr / 85 gr
multi_before = sklavenitis['product_name_clean'].str.extract(multipack_before_pattern)
multi_after = sklavenitis['product_name_clean'].str.extract(multipack_after_pattern)
single = sklavenitis['product_name_clean'].str.extract(single_pattern)
pack_only_x = sklavenitis['product_name_clean'].str.extract(pack_only_x_pattern)
pack_of = sklavenitis['product_name_clean'].str.extract(pack_of_pattern)

# NEW
additive = sklavenitis['product_name_clean'].str.extract(additive_pattern)

# 3. Размер как сырой текст
sklavenitis['size_raw'] = np.select(
    [
        multi_before[0].notna(),
        multi_after[0].notna(),
        additive[0].notna(),   # NEW
        single[0].notna()
    ],
    [
        sklavenitis['product_name_clean'].str.extract(
            r'((?:\d+\s*(?:pcs|sachet(?:s)?)?\s*[x\u00d7*]\s*)\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs)\b)'
        )[0],
        sklavenitis['product_name_clean'].str.extract(
            r'(\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs)\s*[x\u00d7*]\s*\d+\s*(?:pcs)?\b)'
        )[0],
        # NEW
        sklavenitis['product_name_clean'].str.extract(
            r'(\d+(?:\.\d+)?\s*\+\s*\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr)\b)'
        )[0],
        sklavenitis['product_name_clean'].str.extract(
            r'((?<!\d[-–—])(?<!\d [-–—])(?<!\d[-–—] )(?<!\d [-–—] )\b\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs)\b)'
        )[0]
    ],
    default=np.nan
)

# 4. Количество упаковок
sklavenitis['pack_count'] = np.select(
    [
        multi_before[0].notna(),
        multi_after[0].notna(),
        pack_only_x[0].notna(),
        pack_of[0].notna(),
        additive[0].notna()    # NEW
    ],
    [
        multi_before[0],
        multi_after[2],
        pack_only_x[0],
        pack_of[0],
        1                      # NEW: 750+250ml = одна упаковка
    ],
    default=1
)

# 5. Размер одной единицы
sklavenitis['unit_size_value'] = np.select(
    [
        multi_before[1].notna(),
        multi_after[0].notna(),
        additive[0].notna(),   # NEW
        single[0].notna()
    ],
    [
        multi_before[1],
        multi_after[0],
        # NEW: складываем две части 750 + 250
        pd.to_numeric(additive[0], errors='coerce') + pd.to_numeric(additive[1], errors='coerce'),
        single[0]
    ],
    default=np.nan
)

# единицы измерения
# NEW: gr сохраняем как отдельную найденную единицу, а ниже при нормализации приводим ее к g
sklavenitis['unit_size_unit'] = np.select(
    [
        multi_before[2].notna(),
        multi_after[1].notna(),
        additive[2].notna(),   # NEW
        single[1].notna()
    ],
    [
        multi_before[2],
        multi_after[1],
        additive[2],           # NEW
        single[1]
    ],
    default=np.nan
)

# 6. Переводим в числа
sklavenitis['pack_count'] = pd.to_numeric(sklavenitis['pack_count'], errors='coerce')
sklavenitis['unit_size_value'] = pd.to_numeric(sklavenitis['unit_size_value'], errors='coerce')

# 7. Общий размер до нормализации
# NEW: умножаем только явную мультиупаковку (4x250ml / 330ml x 6 / 3 sachets x 8g)
# NEW: если штуки стоят отдельно от общего веса, как в x15pcs 216g, total остается 216g, а не 15 * 216g
sklavenitis['size_total_raw'] = np.select(
    [
        multi_before[0].notna(),
        multi_after[0].notna(),
        additive[0].notna(),
        single[0].notna()
    ],
    [
        sklavenitis['pack_count'] * sklavenitis['unit_size_value'],
        sklavenitis['pack_count'] * sklavenitis['unit_size_value'],
        sklavenitis['unit_size_value'],
        sklavenitis['unit_size_value']
    ],
    default=np.nan
)

# NEW: отдельная корректировка для случаев типа x15pcs 216g
# NEW: если количество штук написано отдельно, а потом идет общий вес, не умножаем total на pack_count
pack_only_with_total_size_mask = (
    sklavenitis['product_name_clean'].str.contains(pack_only_x_pattern, regex=True, na=False)
    & sklavenitis['product_name_clean'].str.contains(single_pattern, regex=True, na=False)
    & ~sklavenitis['product_name_clean'].str.contains(multipack_before_pattern, regex=True, na=False)
    & ~sklavenitis['product_name_clean'].str.contains(multipack_after_pattern, regex=True, na=False)
    & ~sklavenitis['product_name_clean'].str.contains(additive_pattern, regex=True, na=False)
)

sklavenitis.loc[pack_only_with_total_size_mask, 'size_total_raw'] = (
    sklavenitis.loc[pack_only_with_total_size_mask, 'unit_size_value']
)

# 8. Нормализация в g / ml / pcs
def normalize_size(row):
    value = row['size_total_raw']
    unit = row['unit_size_unit']
    
    if pd.isna(value) or pd.isna(unit):
        return pd.Series([np.nan, np.nan])
    
    if unit == 'g':
        return pd.Series([value, 'g'])
    # NEW: короткую запись граммов gr приводим к базовой единице g
    elif unit == 'gr':
        return pd.Series([value, 'g'])
    elif unit == 'kg':
        return pd.Series([value * 1000, 'g'])
    elif unit == 'ml':
        return pd.Series([value, 'ml'])
    elif unit in ['l', 'lt', 'ltr']:
        return pd.Series([value * 1000, 'ml'])
    elif unit == 'pcs':
        return pd.Series([value, 'pcs'])
    else:
        return pd.Series([np.nan, np.nan])

sklavenitis[['size_total_base', 'size_base_unit']] = sklavenitis.apply(normalize_size, axis=1)

# 9. Красивый nullable int для количества упаковок
sklavenitis['pack_count'] = sklavenitis['pack_count'].astype('Int64')

In [ ]:
# удалю еще 2 колонки чтобы не мешались
sklavenitis = sklavenitis.drop(columns=['promo_price', 'promo_label'])
sklavenitis

In [ ]:
# чтобы колонки с числами другие тоже были integer и лучше читались 
# sklavenitis['unit_size_value'] = sklavenitis['unit_size_value'].round().astype('Int64') #не стоит раундить до целого, иначе будет неточным для 1.5 кг
# sklavenitis['size_total_raw'] = sklavenitis['size_total_raw'].round().astype('Int64')
sklavenitis['size_total_base'] = sklavenitis['size_total_base'].round().astype('Int64')

In [ ]:
sklavenitis

## 3. Извлечение grade и сборка product_core

Здесь отдельно достаём grade, а потом убираем из названия служебные части, чтобы получить более чистый product_core.

In [ ]:
# БЫЛО
# ____________________________________________________________________________________________
# # теперь чистим название, чтобы там не было лишних значений, веса и тд 
# # и еще параллельно выделяем отдельной колонкой grade (это было на креветках и осьминогах):
# # 10/20
# # 30/40
# # 06/08
# # t4
# # t6

# # 1. Вытащить grade / size range
# grade_pattern = r'\b(?:\d{1,2}/\d{1,2}|t\d+)\b'

# sklavenitis['grade'] = sklavenitis['product_name_clean'].str.extract(f'({grade_pattern})', expand=False)

# # 2. Создать колонку без размеров/объёмов/pack count/grade
# sklavenitis['product_core'] = sklavenitis['product_name_clean'].copy()

# # убрать multipack before: 4x125g / 4 x 125 g / 3 x 54 pcs
# # общий вид функции: df['col'].str.replace(что_ищем, на_что_заменяем, regex=...)
# sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
#     r'\b\d+\s*x\s*\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs)\b',
#     '',
#     regex=True
# )

# # убрать multipack after: 330ml x 6
# sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
#     r'\b\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs)\s*x\s*\d+\b',
#     '',
#     regex=True
# )

# # убрать обычный размер/объём: 250g / 1lt / 2.2kg / 4pcs
# sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
#     r'\b\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs)\b',
#     '',
#     regex=True
# )

# # убрать pack count: x6 / x 6
# sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
#     r'\bx\s*\d+\b',
#     '',
#     regex=True
# )

# # убрать pack of 2
# sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
#     r'\bpack\s+of\s+\d+\b',
#     '',
#     regex=True
# )

# # убрать grade: 10/20, 30/40, 06/08, t4, t6
# sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
#     grade_pattern,
#     '',
#     regex=True
# )

# # подчистить пробелы
# sklavenitis['product_core'] = (
#     sklavenitis['product_core']
#     .str.replace(r'\s+', ' ', regex=True)
#     .str.strip()
# )


# СТАЛО
# ____________________________________________________________________________________________
# теперь чистим название, чтобы там не было лишних значений, веса и тд 
# и еще параллельно выделяем отдельной колонкой grade (это было на креветках и осьминогах):
# 10/20
# 30/40
# 06/08
# t4
# t6

# 1. Вытащить grade / size range
# percent / percent fat: 0%, 1.5%, 17% fat
# washes: 36 washes
# age / stage / range: 4+ months, 6-12 months, 12 years old
# child range: 4-10 years, 20-35kg
# NEW: range с граммами тоже учитываем в короткой записи через gr, например 85gr или 10-20gr
# dimensions / measures: 42cm x 38cm, 190 meters, 20m, 8 servings, 9volt
# size codes: 6/10, t4, size 2, a4, 5*
# комбинация sheets: 225 sheets, 2ply
# standalone 3ply пока специально не добавляем
grade_percent_pattern = r'\b\d+(?:[\.,]\d+)?%(?:\s*fat)?(?=\b|$)'
grade_washes_pattern = r'\b\d+\s*washes\b'
grade_age_pattern = r'\b(?:age\s*)?\d+\+\s*(?:months?|years?)\b|\b\d+\s*[-–—]\s*\d+\s*(?:months?|years?)\b|\b\d+\s*years?\s*old\b'
grade_measure_pattern = r'\b\d+(?:[\.,]\d+)?\s*(?:meters|m|servings|volt|min)\b'
grade_dimensions_pattern = r'\b\d+(?:[\.,]\d+)?\s*(?:cm|m)?\s*[x\u00d7*]\s*\d+(?:[\.,]\d+)?\s*(?:cm|m)?\b'
grade_range_pattern = r'\b\d+\s*[-–—]\s*\d+\s*(?:g|gr|kg)\b'
grade_slash_pattern = r'\b\d{1,2}/\d{1,2}\b'
grade_t_pattern = r'\bt\d+\b'
grade_size_pattern = r'\bsize\s*\d+\b'
grade_a_pattern = r'\ba\d+\b'
grade_star_pattern = r'\b\d+\*+(?=\s|$)'
grade_sheets_pattern = r'\b\d+\s*sheets(?:\s*,\s*\d+ply)?\b'
grade_intensity_pattern = r'\bintensity\s*\d+\b'
# NEW: отдельно перечисляем все варианты тире, чтобы regex одинаково видел их в названии
grade_dash_pattern = r'[-\u2013\u2014]'

# диапазоны для grade достаём по частям через str.extract
# так нам не важно, стоят ли между ними скобки, запятые или просто пробелы
grade_age_pattern = rf'\b(?:age\s*)?\d+\+\s*(?:months?|years?)\b|\b\d+\s*{grade_dash_pattern}\s*\d+\s*(?:months?|years?)\b|\b\d+\s*years?\s*old\b'
grade_range_pattern = rf'\b\d+\s*{grade_dash_pattern}\s*\d+\s*(?:g|gr|kg)\b'

# здесь лежат остальные устойчивые форматы grade, которые не нужно собирать из двух частей
grade_general_pattern = (
    f'(?:{grade_percent_pattern}|{grade_washes_pattern}|{grade_measure_pattern}|{grade_dimensions_pattern}|'
    f'{grade_slash_pattern}|{grade_t_pattern}|{grade_size_pattern}|{grade_a_pattern}|{grade_star_pattern}|'
    f'{grade_sheets_pattern}|{grade_intensity_pattern})'
)

# этот общий grade_pattern потом используем ниже, когда чистим product_core
grade_pattern = f'(?:{grade_age_pattern}|{grade_range_pattern}|{grade_general_pattern})'

# Шаг 1. Отдельно достаём age range и range с unit
grade_age = sklavenitis['product_name_clean'].str.extract(f'({grade_age_pattern})', expand=False)
grade_range = sklavenitis['product_name_clean'].str.extract(f'({grade_range_pattern})', expand=False)

# Шаг 2. Если нашли обе части, собираем их в один grade
grade_combined = pd.Series(pd.NA, index=sklavenitis.index, dtype='object')
grade_both_mask = grade_age.notna() & grade_range.notna()
grade_combined.loc[grade_both_mask] = grade_age.loc[grade_both_mask].str.strip() + ', ' + grade_range.loc[grade_both_mask].str.strip()
grade_combined = grade_combined.combine_first(grade_age)
grade_combined = grade_combined.combine_first(grade_range)

# Шаг 3. Потом достаём остальные варианты grade
grade_general = sklavenitis['product_name_clean'].str.extract(f'({grade_general_pattern})', expand=False)

# Шаг 4. Сначала берём составной диапазон, а если его нет — остальные форматы
sklavenitis['grade'] = grade_combined.combine_first(grade_general)

# Дальше собираем product_core: берём product_name_clean и постепенно убираем из него служебные части
sklavenitis['product_core'] = sklavenitis['product_name_clean'].copy()

# Сначала убираем multipack вида 4x125g или 3 x 54 pcs
sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
    r'\b\d+\s*(?:pcs|sachet(?:s)?)?\s*[x\u00d7*]\s*\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs)\b',
    '',
    regex=True
)

# Потом убираем запись, где size стоит перед количеством
# было: ...\s*x\s*\d+\b
# стало: ...\s*x\s*\d+\s*(?:pcs)?\b
sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
    r'\b\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs)\s*[x\u00d7*]\s*\d+\s*(?:pcs)?\b',
    '',
    regex=True
)

# NEW: отдельно убираем additive size, например 750+250ml
sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
    r'\b\d+(?:\.\d+)?\s*\+\s*\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr)\b',
    '',
    regex=True
)

# Потом убираем обычный size или объём
sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
    r'(?<!\d[-–—])(?<!\d [-–—])(?<!\d[-–—] )(?<!\d [-–—] )\b\d+(?:\.\d+)?\s*(?:g|gr|kg|ml|l|lt|ltr|pcs)\b',
    '',
    regex=True
)

# Потом убираем отдельный pack count
sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
    r'\b[x\u00d7]\s*\d+\s*(?:pcs)?\b',
    '',
    regex=True
)

# NEW: если число уже удалилось, убираем оставшийся pcs
sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
    r'\bpcs\b',
    '',
    regex=True
)

# Потом убираем конструкции вида pack of 2
sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
    r'\bpack\s+of\s+\d+\b',
    '',
    regex=True
)

# После этого убираем из product_core всё, что уже записали в grade
sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
    grade_pattern,
    '',
    regex=True
)

# NEW: если осталось одинокое approx, тоже убираем его
sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
    r'\bapprox\.?\b',
    '',
    regex=True
)

# В самом конце подчищаем хвосты после всех удалений
# NEW: сначала убираем пустые и одинокие скобки
sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
    r'\(\s*\)|\s*\)\s*$|\(\s*$',
    '',
    regex=True
)

# NEW: потом убираем хвостовые тире
sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
    r'\s*[-\u2013\u2014]\s*$',
    '',
    regex=True
)

# NEW: и ещё раз проверяем скобки после тире
sklavenitis['product_core'] = sklavenitis['product_core'].str.replace(
    r'\(\s*\)|\s*\)\s*$|\(\s*$',
    '',
    regex=True
)

sklavenitis['product_core'] = (
    sklavenitis['product_core']
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

In [ ]:
sklavenitis

In [ ]:
# посмотреть как выглядят строки с кг теперь 
sklavenitis[
    sklavenitis['product_name_clean'].str.contains(r'\b\d+(?:\.\d+)?\s*kg\b', regex=True, na=False)
][['product_name_clean', 'pack_count', 'unit_size_value', 'unit_size_unit', 'size_total_base', 'size_base_unit']]

## 4. Подготовка выборки для ручной разметки

Этот блок нужен для создания файлов, которые потом размечаются вручную и используются для словарей.

In [ ]:
# для разметки в будущем под NLP мини-часть датасета
label_df = sklavenitis[['product_core']].sample(30, random_state=42).copy()
# label_df.to_excel(r'.\label_df.xlsx', index=False)

### Правила разметки:

### Правило 1. brand
Только сам бренд, без типа товара.

Если:
macedonikos halva
то brand = macedonikos, а не macedonikos halva

### Правило 2. product_type
Главный тип товара.

Если:
halva with almonds
то product_type = halva

### Правило 3. variant
То, что отличает товар внутри типа, но не является общим атрибутом.

Например:
- cocoa
- almonds
- banana

### Правило 4. attributes
Общие свойства.

Например:
- organic
- vegan
- gluten free
- smoked

### Правило 5. grade

Размерный класс / диапазон:
- 10/20
- t6

### Правило 6. model_code

Технический код товара:
- ob32
- dl2016b2

In [ ]:
# для разметки в будущем под NLP мини-часть датасета С ИНДЕКСАМИ 
label_df_with_index = sklavenitis[['product_core']].sample(100, random_state=42).copy()
# label_df_with_index.to_excel(r'.\label_df_with_index.xlsx', index=True)

In [ ]:
sklavenitis_raw = pd.read_csv(base_path / "sklavenitis-ypsonas.csv", encoding="utf-8")

In [ ]:
label_df_with_index = label_df_with_index.join(sklavenitis_raw[['product_name', 'image_url', 'category_path']], how='left') 

In [ ]:
# label_df_with_index.to_excel(r'.\label_df_with_index.xlsx', index=True)

In [ ]:
# посмотреть примеры с пустыми скобками в конце
sklavenitis[
    sklavenitis['product_core'].str.contains(r'\s+\(\)\s*$', regex=True, na=False)
][['product_core']]

# посмотреть примеры с точкой в конце
sklavenitis[
    sklavenitis['product_core'].str.contains(r'\.\s*$', regex=True, na=False)
][['product_core']]

# убрать пустые скобки в конце и точки
sklavenitis['product_core'] = (
    sklavenitis['product_core']
    .str.replace(r'\s+\(\)\s*$', '', regex=True)  # убрать пустые скобки в конце: " product ()"
    .str.replace(r'\.\s*$', '', regex=True)       # убрать точку в конце
    .str.strip()
)

In [ ]:
# видим, что после strip появились снова, убираем по кругу 
sklavenitis['product_core'] = (
    sklavenitis['product_core']
    .str.replace(r'\s+\(\)\s*$', '', regex=True)  # убрать пустые скобки в конце: " product ()"
    .str.replace(r'\.\s*$', '', regex=True)       # убрать точку в конце
    .str.strip()
)

In [ ]:
# посмотреть в конце дефисы лишние
sklavenitis[
    sklavenitis['product_core'].str.contains(r'\s+[-–—]\s*$', regex=True, na=False)
][['product_core']]

In [ ]:
# удалить в конце разного вида дефисы
sklavenitis['product_core'] = (
    sklavenitis['product_core']
    .str.replace(r'\s+[-–—]\s*$', '', regex=True)
    .str.strip()
)

In [ ]:
# # импортировать обновленный датасет с 100 размеченными вариантами обратно
# label_updated = pd.read_excel(
#     r'.\label_df_with_index.xlsx',
#     index_col=0
# )

In [ ]:
# label_updated 

In [ ]:
# # посмотреть примеры с пустыми скобками в конце
# label_updated[
#     label_updated['product_core'].str.contains(r'\s+\(\)\s*$', regex=True, na=False)
# ][['product_core']]

# # посмотреть примеры с точкой в конце
# label_updated[
#     label_updated['product_core'].str.contains(r'\.\s*$', regex=True, na=False)
# ][['product_core']]

# # убрать пустые скобки в конце и точки
# label_updated['product_core'] = (
#     label_updated['product_core']
#     .str.replace(r'\s+\(\)\s*$', '', regex=True)  # убрать пустые скобки в конце: " product ()"
#     .str.replace(r'\.\s*$', '', regex=True)       # убрать точку в конце
#     .str.strip()
# )

In [ ]:
# # посмотреть в конце дефисы лишние
# label_updated[
#     label_updated['product_core'].str.contains(r'\s+[-–—]\s*$', regex=True, na=False)
# ][['product_core']]

# # удалить в конце разного вида дефисы
# label_updated['product_core'] = (
#     label_updated['product_core']
#     .str.replace(r'\s+[-–—]\s*$', '', regex=True)
#     .str.strip()
# )

In [ ]:
sklavenitis_for_annotation_100_500=sklavenitis.join(sklavenitis_raw[['image_url']], how='left') 

In [ ]:
sklavenitis_for_annotation_100_500

In [ ]:
sklavenitis_for_annotation_100_500.loc[2536]

In [ ]:
sklavenitis_for_annotation_100_500.loc[2405]

In [ ]:
# строки, где в product_core остался отдельный pcs
sklavenitis_for_annotation_100_500.loc[905, ['product_name', 'product_name_clean', 'product_core']]

In [ ]:
# строки, где в product_core остался approx или approx.
approx_rows = sklavenitis_for_annotation_100_500[
    sklavenitis_for_annotation_100_500['product_core'].str.contains(r'\bapprox\.?\b', regex=True, na=False)
][['product_name', 'product_name_clean', 'product_core', 'size_raw', 'pack_count', 'grade']]

approx_rows

In [ ]:
sklavenitis_for_annotation_100_500.loc[2536]

## 5. Проверка полного обработанного датасета

Здесь начинаем смотреть на полный sklavenitis, уже обработанный после regex-очистки.

In [ ]:
# Это финальный очищенный файл по полному датасету Sklavenitis.
# В нем уже собран конечный очищенный product_core и вынесены то, что удалось стабильно достать в отдельные колонки.
# Дальше в ноутбуке ориентируемся именно на этот файл как на основной источник правды по очищенному product_core.
processed_sklavenitis=pd.read_csv("output/sklavenitis_ypsonas_processed.csv")
processed_sklavenitis.head()

In [ ]:
processed_sklavenitis.loc[2536]

In [ ]:
processed_sklavenitis.head(50)

## 6. Базовый словарный проход по brand и product_type

Здесь возвращаем тот базовый шаг, на котором мы уже остановились раньше:
- собираем словари из reviewed-разметки
- ищем `brand` по всему `product_core`, не только в начале
- ищем `product_type` в тексте после удаления найденного бренда
- считаем coverage

Здесь **нет** дополнительной логики с `match_mode`, `ambiguous` и спец-правилами для отдельных брендов.
Это просто базовый словарный проход, который мы пока фиксируем как есть.


## 1. Импорт и пути

Здесь подключаем библиотеки и задаём пути к входным и выходным файлам.


In [ ]:
import re
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
OUTPUT_DIR = ROOT / 'output'

# Финальный reviewed-файл для словарей и coverage.
# Это canonical-версия reviewed-разметки, уже привязанная к конечному очищенному product_core.
REVIEWED_PATH = OUTPUT_DIR / 'label_df_random42_labeled_500_reviewed_v5_canonical.csv'

# Финальный очищенный файл по полному датасету Sklavenitis.
# Именно он считается главным источником правды для product_core после очистки.
PROCESSED_PATH = OUTPUT_DIR / 'sklavenitis_ypsonas_processed.csv'

BRAND_VOCAB_PATH = OUTPUT_DIR / 'brand_vocab.csv'
PRODUCT_TYPE_VOCAB_PATH = OUTPUT_DIR / 'product_type_vocab.csv'
FIRST_PASS_PATH = OUTPUT_DIR / 'sklavenitis_brand_product_type_first_pass.csv'
COVERAGE_PATH = OUTPUT_DIR / 'brand_product_type_coverage_summary.csv'
UNKNOWN_BRANDS_PATH = OUTPUT_DIR / 'unknown_brand_product_core_top.csv'
UNKNOWN_PRODUCT_TYPES_PATH = OUTPUT_DIR / 'unknown_product_type_product_core_top.csv'
BRAND_NOT_FIRST_WORD_PATH = OUTPUT_DIR / 'brand_match_not_first_word_examples.csv'

PRODUCT_TYPE_NON_LITERAL_CASES_PATH = OUTPUT_DIR / 'product_type_non_literal_cases.csv'
PRODUCT_TYPE_TRIGGER_CANDIDATES_PATH = OUTPUT_DIR / 'product_type_trigger_candidates.csv'
PRODUCT_TYPE_REPEATED_TRIGGER_CANDIDATES_PATH = OUTPUT_DIR / 'product_type_trigger_candidates_repeated.csv'
PRODUCT_TYPE_TRIGGER_CANDIDATES_XLSX_PATH = OUTPUT_DIR / 'product_type_trigger_candidates.xlsx'
PRODUCT_TYPE_REPEATED_TRIGGER_CANDIDATES_XLSX_PATH = OUTPUT_DIR / 'product_type_trigger_candidates_repeated.xlsx'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REVIEWED_PATH, PROCESSED_PATH


## 2. Загружаем данные

Здесь используем уже финальные файлы, а не старые промежуточные версии.

- `sklavenitis_ypsonas_processed.csv` — это конечный очищенный файл по полному датасету Sklavenitis.
  В нем уже собран очищенный `product_core` и вынесены устойчиво найденные `size`, `grade` и другие служебные поля.
- `label_df_random42_labeled_500_reviewed_v5_canonical.csv` — это reviewed-разметка на 500 строк,
  уже джойнутая к финальному очищенному файлу, чтобы словари и coverage считались от той же версии `product_core`.


In [ ]:
# Загружаем уже канонические файлы: reviewed после join к финальному processed и сам полный cleaned processed.
reviewed = pd.read_csv(REVIEWED_PATH)
processed = pd.read_csv(PROCESSED_PATH)

print('reviewed rows:', len(reviewed))
print('processed rows:', len(processed))

reviewed[['product_core', 'brand', 'product_type', 'variant', 'attributes', 'grade', 'pieces']].head()


## 3. Нормализация текста

Приводим текст к более единообразному виду: убираем лишние пробелы,
делаем lowercase и заменяем пропуски на пустую строку.


In [ ]:
def normalize_text(value):
    if pd.isna(value):
        return ''
    text = str(value).lower()
    text = re.sub(r'[\u200b\u200c\u200d\ufeff]', '', text)
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    return text


reviewed = reviewed.copy()
processed = processed.copy()

reviewed['product_core'] = reviewed['product_core'].fillna('')
reviewed['brand'] = reviewed['brand'].map(normalize_text)
reviewed['product_type'] = reviewed['product_type'].map(normalize_text)
reviewed['product_core_norm'] = reviewed['product_core'].map(normalize_text)

processed['product_core'] = processed['product_core'].fillna('')
processed['product_core_norm'] = processed['product_core'].map(normalize_text)

reviewed[['product_core', 'product_core_norm', 'brand', 'product_type']].head()


## 4. Собираем словари brand и product_type

Это обзорные словари из reviewed-разметки:
- само значение
- сколько раз оно встретилось
- один пример `product_core`


In [ ]:
def build_vocab_table(df, column):
    non_empty = df.loc[df[column].ne(''), ['product_core', column]].copy()
    summary = (
        non_empty.groupby(column, as_index=False)
        .agg(
            count=('product_core', 'size'),
            example_product_core=('product_core', 'first')
        )
        .sort_values(['count', column], ascending=[False, True], ignore_index=True)
    )
    return summary


brand_vocab = build_vocab_table(reviewed, 'brand')
product_type_vocab = build_vocab_table(reviewed, 'product_type')

print('distinct brands:', len(brand_vocab))
print('distinct product_types:', len(product_type_vocab))
brand_vocab.head(10)


In [ ]:
product_type_vocab.head(10)


## 5. Первый автоматический проход по полному sklavenitis

Базовая логика:
- `brand` ищем по всей строке, не только в начале
- многословные бренды тоже ищем
- после найденного бренда убираем его из текста один раз
- `product_type` ищем в оставшемся тексте
- если `product_type` состоит из нескольких слов, разрешаем небольшие вставки между ними


In [ ]:
def build_phrase_pattern(term):
    escaped = re.escape(term)
    return re.compile(rf'(?<![a-z0-9]){escaped}(?![a-z0-9])')


def build_flexible_multiword_pattern(term, max_gap_words=3):
    words = term.split()
    if len(words) <= 1:
        return build_phrase_pattern(term)

    gap = rf"(?:[\s\-/,()]+[a-z0-9%'.&+]+){{0,{max_gap_words}}}[\s\-/,()]+"
    pattern = rf'(?<![a-z0-9]){re.escape(words[0])}'

    for word in words[1:]:
        pattern += gap + re.escape(word)

    pattern += r'(?![a-z0-9])'
    return re.compile(pattern)


def match_brand_from_vocab(product_core_norm, brand_terms):
    if not product_core_norm:
        return '', None

    best_brand = ''
    best_start = None

    for brand in brand_terms:
        match = build_phrase_pattern(brand).search(product_core_norm)
        if not match:
            continue

        if not best_brand:
            best_brand = brand
            best_start = match.start()
            continue

        is_longer = len(brand) > len(best_brand)
        same_length_earlier = (
            len(brand) == len(best_brand)
            and best_start is not None
            and match.start() < best_start
        )

        if is_longer or same_length_earlier:
            best_brand = brand
            best_start = match.start()

    return best_brand, best_start


def remove_brand_once(product_core_norm, brand):
    if not product_core_norm or not brand:
        return product_core_norm
    cleaned = build_phrase_pattern(brand).sub(' ', product_core_norm, count=1)
    cleaned = re.sub(r'\s+', ' ', cleaned)
    return cleaned.strip(' -')


def match_product_type_from_vocab(core_without_brand, product_type_terms):
    if not core_without_brand:
        return '', ''

    best_match = ''
    best_start = None
    best_method = ''

    for product_type in product_type_terms:
        exact_match = build_phrase_pattern(product_type).search(core_without_brand)
        flexible_match = None

        if len(product_type.split()) > 1:
            flexible_match = build_flexible_multiword_pattern(product_type).search(core_without_brand)

        match = exact_match or flexible_match
        if not match:
            continue

        current_method = (
            'product_type_vocab_phrase'
            if exact_match is not None
            else 'product_type_vocab_flexible'
        )

        if not best_match:
            best_match = product_type
            best_start = match.start()
            best_method = current_method
            continue

        is_longer = len(product_type) > len(best_match)
        same_length_earlier = (
            len(product_type) == len(best_match)
            and best_start is not None
            and match.start() < best_start
        )

        if is_longer or same_length_earlier:
            best_match = product_type
            best_start = match.start()
            best_method = current_method

    return best_match, best_method


brand_terms = brand_vocab['brand'].tolist()
product_type_terms = product_type_vocab['product_type'].tolist()

brand_predictions = []
brand_methods = []
brand_match_starts = []
core_without_brand_values = []
product_type_predictions = []
product_type_methods = []

for row in processed.itertuples(index=False):
    product_core_norm = row.product_core_norm

    brand_pred, brand_start = match_brand_from_vocab(product_core_norm, brand_terms)
    if brand_pred:
        brand_method = (
            'brand_vocab_first_word'
            if brand_start == 0
            else 'brand_vocab_not_first_word'
        )
    else:
        brand_method = ''

    core_without_brand = remove_brand_once(product_core_norm, brand_pred)
    product_type_pred, product_type_method = match_product_type_from_vocab(
        core_without_brand, product_type_terms
    )

    brand_predictions.append(brand_pred)
    brand_methods.append(brand_method)
    brand_match_starts.append(brand_start)
    core_without_brand_values.append(core_without_brand)
    product_type_predictions.append(product_type_pred)
    product_type_methods.append(product_type_method)

processed['brand_pred'] = brand_predictions
processed['brand_match_method'] = brand_methods
processed['brand_match_start'] = brand_match_starts
processed['product_core_without_brand'] = core_without_brand_values
processed['product_type_pred'] = product_type_predictions
processed['product_type_match_method'] = product_type_methods

processed[[
    'product_core', 'brand_pred', 'brand_match_method', 'brand_match_start',
    'product_core_without_brand', 'product_type_pred', 'product_type_match_method'
]].head(15)


## 6. Coverage

Считаем, сколько строк удалось покрыть на базовой логике.


In [ ]:
coverage = pd.DataFrame([
    {
        'field': 'brand',
        'total_rows': len(processed),
        'matched_rows': int(processed['brand_pred'].ne('').sum()),
        'matched_share': round(processed['brand_pred'].ne('').mean(), 4),
        'vocab_match': int(processed['brand_pred'].ne('').sum()),
        'brand_match_first_word': int((processed['brand_match_method'] == 'brand_vocab_first_word').sum()),
        'brand_match_not_first_word': int((processed['brand_match_method'] == 'brand_vocab_not_first_word').sum()),
        'unknown_rows': int(processed['brand_pred'].eq('').sum())
    },
    {
        'field': 'product_type',
        'total_rows': len(processed),
        'matched_rows': int(processed['product_type_pred'].ne('').sum()),
        'matched_share': round(processed['product_type_pred'].ne('').mean(), 4),
        'vocab_match': int(processed['product_type_pred'].ne('').sum()),
        'flexible_multiword_match': int((processed['product_type_match_method'] == 'product_type_vocab_flexible').sum()),
        'unknown_rows': int(processed['product_type_pred'].eq('').sum())
    }
])

coverage


## 7. Unknown cases

Эти таблицы полезны для ручной проверки:
- где не поймался `brand`
- где не поймался `product_type`


In [ ]:
unknown_brand = (
    processed.loc[processed['brand_pred'].eq(''), ['product_core', 'product_name']]
    .groupby('product_core', as_index=False)
    .agg(
        count=('product_name', 'size'),
        example_product_name=('product_name', 'first')
    )
    .sort_values(['count', 'product_core'], ascending=[False, True], ignore_index=True)
)

unknown_product_type = (
    processed.loc[processed['product_type_pred'].eq(''), ['product_core', 'product_name']]
    .groupby('product_core', as_index=False)
    .agg(
        count=('product_name', 'size'),
        example_product_name=('product_name', 'first')
    )
    .sort_values(['count', 'product_core'], ascending=[False, True], ignore_index=True)
)

print('unknown brand rows:', int(processed['brand_pred'].eq('').sum()))
print('unknown product_type rows:', int(processed['product_type_pred'].eq('').sum()))


In [ ]:
unknown_brand.head(20)


In [ ]:
unknown_product_type.head(20)


## 8. Кейсы, где бренд поймался не первым словом

Это просто проверочная таблица, чтобы можно было глазами посмотреть,
насколько адекватно работает поиск `brand` по всей строке.


In [ ]:
brand_not_first_word = (
    processed.loc[
        (processed['brand_pred'].ne(''))
        & (processed['brand_match_method'] == 'brand_vocab_not_first_word'),
        ['brand_pred', 'product_core', 'product_name', 'brand_match_start']
    ]
    .groupby(['brand_pred', 'product_core'], as_index=False)
    .agg(
        count=('product_name', 'size'),
        first_brand_position=('brand_match_start', 'min'),
        example_product_name=('product_name', 'first')
    )
    .sort_values(['count', 'brand_pred', 'product_core'], ascending=[False, True, True], ignore_index=True)
)

print('brand matched not as first word:', len(processed.loc[processed['brand_match_method'] == 'brand_vocab_not_first_word']))
brand_not_first_word.head(30)


In [ ]:
processed.loc[
    processed['brand_match_method'] == 'brand_vocab_not_first_word',
    ['product_core', 'brand_pred', 'brand_match_start', 'product_type_pred', 'product_name']
].head(30)


## 9. Сохраняем базовые результаты

После выполнения этой ячейки в папке `output` появятся базовые CSV по brand/product_type coverage.


In [ ]:
brand_vocab.to_csv(BRAND_VOCAB_PATH, index=False, encoding='utf-8-sig')
product_type_vocab.to_csv(PRODUCT_TYPE_VOCAB_PATH, index=False, encoding='utf-8-sig')
processed.to_csv(FIRST_PASS_PATH, index=False, encoding='utf-8-sig')
coverage.to_csv(COVERAGE_PATH, index=False, encoding='utf-8-sig')
unknown_brand.to_csv(UNKNOWN_BRANDS_PATH, index=False, encoding='utf-8-sig')
unknown_product_type.to_csv(UNKNOWN_PRODUCT_TYPES_PATH, index=False, encoding='utf-8-sig')
brand_not_first_word.to_csv(BRAND_NOT_FIRST_WORD_PATH, index=False, encoding='utf-8-sig')

print('saved:', BRAND_VOCAB_PATH)
print('saved:', PRODUCT_TYPE_VOCAB_PATH)
print('saved:', FIRST_PASS_PATH)
print('saved:', COVERAGE_PATH)
print('saved:', UNKNOWN_BRANDS_PATH)
print('saved:', UNKNOWN_PRODUCT_TYPES_PATH)
print('saved:', BRAND_NOT_FIRST_WORD_PATH)


## 10. Углубляемся в product_type триггеры

Теперь отдельным следующим шагом идём глубже именно в `product_type`.
Здесь `brand` используется только как вспомогательная подсказка внутри reviewed-500,
потому что мы хотим понять, какие части названия уже размечены,
а какие куски текста ещё могут нести смысл `product_type`.

Новая логика здесь такая:
- сначала проверяем, есть ли `product_type` прямо в `product_core`
- если есть, это обычный literal-кейс, и он не так интересен для триггеров
- если нет, только тогда вычитаем из `product_core` уже размеченные части
- смотрим, что осталось после вычитания
- если остаток не пустой, используем его как источник кандидатов на триггеры
- если остаток пустой, сохраняем такие кейсы отдельно


## 11. Сначала делим кейсы на literal и non-literal

`literal` здесь значит: нужный `product_type` прямо написан в `product_core`.

Пример literal-кейса:
- `product_core = greek yogurt strawberry`
- `product_type = yogurt`
- слово `yogurt` в тексте есть

`non-literal` значит: правильный `product_type` известен из ручной разметки,
но в самом `product_core` он не написан теми же словами.

Пример non-literal-кейса:
- `product_core = aged graviera with thyme`
- `product_type = cheese`
- слова `cheese` в тексте нет

Только для non-literal кейсов мы дальше вычитаем уже размеченные части:
- `brand`
- `variant`
- `attributes`

Важно: `No.`, `No5`, `No1` и похожие обозначения теперь не считаем `grade` на этом шаге.
Они остаются внутри `product_core` и будут разруливаться уже следующим алгоритмом отдельно.

После этого смотрим, что осталось.
Если ничего не осталось, это тоже важный отдельный тип кейсов.


In [ ]:
# Разбиваем текст на токены, чтобы потом безопасно вычитать целые фразы.
TOKEN_PATTERN = re.compile(r"[a-z]+|\d+")

# Это только лёгкая уборка служебных слов-связок.
# Это уже не основная логика блока.
LIGHT_CONNECTOR_WORDS = {'and', 'with', 'for', 'from', 'the', 'a', 'an', 'to', 'of', 'in', 'on', 'by', 'or', 'at'}


def tokenize(text):
    # Приводим текст к нижнему регистру, убираем zero-width символы
    # и разбиваем на чистый список токенов.
    cleaned = re.sub(r'[\u200b\u200c\u200d\ufeff]', '', str(text).lower())
    return TOKEN_PATTERN.findall(cleaned)


def normalize_phrase_list(value):
    # В variant / attributes может быть несколько фраз через запятую.
    # Разбиваем их, чтобы потом вычитать каждую фразу отдельно.
    if not value:
        return []
    parts = [normalize_text(part) for part in str(value).split(',')]
    return [part for part in parts if part]


def build_grade_removal_candidates(value):
    # Для grade иногда нужен не только исходный текст, но и его слитная форма.
    # Пример: grade = '12-24m', а в product_core осталось '1224m'.
    candidates = []
    for part in normalize_phrase_list(value):
        candidates.append(part)
        compact = re.sub(r'[\s\-–—/]+', '', part)
        if compact and compact != part:
            candidates.append(compact)
    # Сохраняем порядок и убираем дубликаты.
    return list(dict.fromkeys(candidates))


def remove_phrase_if_present(text, phrase):
    # Вспомогательная функция: если точная фраза есть в тексте, вычитаем её один раз.
    if not phrase:
        return text
    if not literal_phrase_present(text, phrase):
        return text
    remaining_tokens = remove_phrase_once_by_tokens(text, phrase)
    return ' '.join(remaining_tokens)


def build_product_core_for_trigger(row):
    # reviewed product_core местами всё ещё содержит grade из старой версии разметки.
    # Здесь делаем локально очищенную версию product_core именно для триггерного шага.
    # Важно: убираем только то, что реально лежит в колонке grade.
    # Обозначения вида No.5 / No1 / No 6 теперь не считаем grade и не вычитаем отсюда автоматически.
    text = row.product_core_norm
    for grade_part in build_grade_removal_candidates(getattr(row, 'grade', '')):
        text = remove_phrase_if_present(text, grade_part)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def literal_phrase_present(text, phrase):
    # Точное вхождение по токенам: слова должны идти подряд и в том же порядке.
    text_joined = ' '.join(tokenize(text))
    phrase_joined = ' '.join(tokenize(phrase))
    return bool(phrase_joined) and phrase_joined in text_joined


def product_type_present_in_core(core_text, product_type):
    # Для product_type используем более мягкую проверку.
    # Сначала ищем точную фразу.
    # Если product_type состоит из нескольких слов,
    # считаем совпадение и тогда, когда все слова есть в product_core,
    # даже если порядок другой или между ними есть другие слова.
    core_tokens = tokenize(core_text)
    product_type_tokens = tokenize(product_type)

    if not product_type_tokens:
        return False

    core_joined = ' '.join(core_tokens)
    product_type_joined = ' '.join(product_type_tokens)

    if product_type_joined in core_joined:
        return True

    if len(product_type_tokens) == 1:
        return False

    core_counts = Counter(core_tokens)
    product_type_counts = Counter(product_type_tokens)
    return all(core_counts[token] >= count for token, count in product_type_counts.items())


def remove_phrase_once_by_tokens(text, phrase):
    # Убираем из текста одну целую фразу, а не отдельные буквы или символы.
    text_tokens = tokenize(text)
    phrase_tokens = tokenize(phrase)

    if not phrase_tokens:
        return text_tokens

    result = []
    i = 0
    n = len(phrase_tokens)

    while i < len(text_tokens):
        if text_tokens[i:i + n] == phrase_tokens:
            i += n
        else:
            result.append(text_tokens[i])
            i += 1

    return result


def subtract_labeled_parts(row):
    # Для non-literal кейсов вычитаем из product_core уже размеченные части:
    # brand, variant и attributes.
    remaining_text = row.product_core_for_trigger
    removed_parts = []

    for column in ['brand', 'variant', 'attributes']:
        for phrase in normalize_phrase_list(getattr(row, column, '')):
            if literal_phrase_present(remaining_text, phrase):
                remaining_tokens = remove_phrase_once_by_tokens(remaining_text, phrase)
                remaining_text = ' '.join(remaining_tokens)
                removed_parts.append(f'{column}: {phrase}')

    # После вычитания делаем только лёгкую уборку служебных слов и пробелов.
    remaining_tokens = [token for token in tokenize(remaining_text) if token not in LIGHT_CONNECTOR_WORDS]
    residual_text = ' '.join(remaining_tokens).strip()

    return residual_text, ' | '.join(removed_parts)


reviewed_product_type_check_rows = []

for row in reviewed.itertuples(index=False):
    if not row.product_type:
        continue

    product_core_for_trigger = build_product_core_for_trigger(row)

    # Шаг 1: проверяем, найден ли product_type в очищенном product_core.
    # Для многословного product_type разрешаем и случай,
    # когда все слова есть в product_core не подряд.
    literal_in_product_core = product_type_present_in_core(product_core_for_trigger, row.product_type)

    # Шаг 2: только для non-literal кейсов строим остаток после вычитания.
    if literal_in_product_core:
        residual_after_subtraction = ''
        removed_parts = ''
    else:
        residual_after_subtraction, removed_parts = subtract_labeled_parts(row)

    reviewed_product_type_check_rows.append({
        'product_core_original': row.product_core,
        'product_core': product_core_for_trigger,
        'product_type': row.product_type,
        'brand': row.brand,
        'variant': getattr(row, 'variant', ''),
        'attributes': getattr(row, 'attributes', ''),
        'grade': getattr(row, 'grade', ''),
        'category_path': getattr(row, 'category_path', ''),
        'literal_in_product_core': literal_in_product_core,
        'removed_parts': removed_parts,
        'residual_after_subtraction': residual_after_subtraction,
        'empty_after_subtraction': (not literal_in_product_core) and (residual_after_subtraction == ''),
    })

reviewed_product_type_check = pd.DataFrame(reviewed_product_type_check_rows)

non_literal_cases = reviewed_product_type_check.loc[
    ~reviewed_product_type_check['literal_in_product_core']
].copy()

non_literal_residual_cases = non_literal_cases.loc[
    non_literal_cases['residual_after_subtraction'].ne('')
].copy()

empty_after_subtraction_cases = non_literal_cases.loc[
    non_literal_cases['empty_after_subtraction']
].copy()

print('rows with product_type:', len(reviewed_product_type_check))
print('literal product_type in product_core:', int(reviewed_product_type_check['literal_in_product_core'].sum()))
print('non-literal cases:', len(non_literal_cases))
print('non-literal with residual text:', len(non_literal_residual_cases))
print('non-literal empty after subtraction:', len(empty_after_subtraction_cases))

non_literal_cases.head(15)


## 12. Из остатка собираем черновые trigger-кандидаты

Основная идея такая:
- сначала вычесть из `product_core` уже размеченные части
- потом посмотреть именно на остаток
- и уже из этого остатка собирать кандидатов на `trigger -> product_type`

То есть теперь триггеры ищутся не в почти сыром названии,
а в том, что осталось неразобранным после вычитания `brand`, `variant` и `attributes`.

Важно: это всё ещё не финальный словарь,
а аккуратный черновик для ручного просмотра.


In [ ]:
def build_trigger_candidates(non_literal_df):
    # ngram_total: сколько раз триггер встретился вообще среди residual-кейсов.
    ngram_total = Counter()
    # ngram_type: сколько раз этот же триггер встретился именно с этим product_type.
    ngram_type = Counter()
    # examples: сохраняем несколько примеров очищенного остатка для ручной проверки.
    examples = defaultdict(list)
    # product_core_examples: сохраняем несколько примеров исходного product_core для того же trigger -> product_type.
    product_core_examples = defaultdict(list)

    for row in non_literal_df.itertuples(index=False):
        row_tokens = tokenize(row.residual_after_subtraction)
        # seen не даёт считать один и тот же триггер несколько раз внутри одной строки.
        seen = set()

        # Смотрим одиночные слова, пары слов и тройки слов.
        # Это помогает ловить и короткие триггеры вроде 'gouda', и фразы вроде 'baby milk'.
        for n in (1, 2, 3):
            for i in range(len(row_tokens) - n + 1):
                gram_tokens = row_tokens[i:i + n]

                # Пропускаем слишком слабые куски: чистые цифры и очень короткие токены.
                if any(token.isdigit() for token in gram_tokens):
                    continue
                if any(len(token) < 3 for token in gram_tokens):
                    continue

                gram = ' '.join(gram_tokens)
                if gram in seen:
                    continue
                seen.add(gram)

                ngram_total[gram] += 1
                ngram_type[(gram, row.product_type)] += 1

                if len(examples[(gram, row.product_type)]) < 3:
                    examples[(gram, row.product_type)].append(row.residual_after_subtraction)
                if len(product_core_examples[(gram, row.product_type)]) < 3:
                    product_core_examples[(gram, row.product_type)].append(row.product_core)

    rows = []
    for (gram, product_type), support in ngram_type.items():
        total = ngram_total[gram]
        precision = support / total
        rows.append({
            'product_type': product_type,
            'trigger': gram,
            # support_in_reviewed = сколько reviewed-строк подтверждают связь trigger -> product_type.
            'support_in_reviewed': support,
            # total_trigger_count = сколько раз этот trigger встретился вообще среди residual-кейсов.
            'total_trigger_count': total,
            # precision_in_residual_subset = support / total.
            'precision_in_residual_subset': round(precision, 3),
            # Если support == 1, триггер может быть полезным, но пока слабо подтверждён.
            'needs_manual_review': support == 1,
            'product_core': ' | '.join(product_core_examples[(gram, product_type)]),
            'example_residual_text': ' | '.join(examples[(gram, product_type)]),
        })

    result = pd.DataFrame(rows).sort_values(
        ['support_in_reviewed', 'precision_in_residual_subset', 'product_type', 'trigger'],
        ascending=[False, False, True, True],
        ignore_index=True,
    )
    return result


product_type_trigger_candidates = build_trigger_candidates(non_literal_residual_cases)
repeated_trigger_candidates = product_type_trigger_candidates.loc[
    (product_type_trigger_candidates['support_in_reviewed'] >= 2)
    & (product_type_trigger_candidates['precision_in_residual_subset'] >= 0.9)
].copy()

print('all trigger candidates from residual text:', len(product_type_trigger_candidates))
print('repeated trigger candidates:', len(repeated_trigger_candidates))
repeated_trigger_candidates.head(20)


## 13. Что смотреть глазами

Ниже самые полезные таблицы:
- `non_literal_cases` — все кейсы, где `product_type` не нашёлся прямо в очищенном `product_core`
- `empty_after_subtraction_cases` — non-literal кейсы, где после вычитания уже размеченных частей ничего не осталось
- `repeated_trigger_candidates` — более устойчивые кандидаты на триггеры

Как это читать:
- `literal_in_product_core = True` значит, что `product_type` нашёлся в очищенном `product_core`: либо точной фразой, либо по всем словам многословного product_type
- `product_core_original` — исходный reviewed `product_core` до локальной дочистки только тех частей, которые лежат в `grade`
- `residual_after_subtraction` — это текстовый остаток после вычитания `brand`, `variant` и `attributes`
- `empty_after_subtraction = True` значит, что non-literal кейс есть, но после вычитания остаток пустой
- высокий `support_in_reviewed` = триггер уже несколько раз подтверждался в reviewed-разметке
- высокий `precision_in_residual_subset` = триггер выглядит более чистым и менее двусмысленным в residual-кейсах


In [ ]:
non_literal_cases[[
    'product_core', 'product_type', 'brand', 'variant', 'attributes',
    'removed_parts', 'residual_after_subtraction', 'empty_after_subtraction'
]].head(20)


In [ ]:
empty_after_subtraction_cases[[
    'product_core', 'product_type', 'brand', 'variant', 'attributes', 'removed_parts'
]].head(20)


In [ ]:
repeated_trigger_candidates.head(30)


## 14. Сохраняем таблицы по product_type триггерам

Эти CSV нужны для следующего ручного шага.
Теперь отдельно сохраняем:
- полную таблицу literal/non-literal проверки
- non-literal кейсы с остатком
- non-literal кейсы с пустым остатком
- все trigger-кандидаты
- более устойчивые repeated trigger-кандидаты


In [ ]:
PRODUCT_TYPE_LITERAL_CHECK_PATH = OUTPUT_DIR / 'product_type_literal_check.csv'
PRODUCT_TYPE_NON_LITERAL_CASES_PATH = OUTPUT_DIR / 'product_type_non_literal_cases.csv'
PRODUCT_TYPE_NON_LITERAL_RESIDUAL_CASES_PATH = OUTPUT_DIR / 'product_type_non_literal_residual_cases.csv'
PRODUCT_TYPE_EMPTY_AFTER_SUBTRACTION_PATH = OUTPUT_DIR / 'product_type_empty_after_subtraction_cases.csv'
PRODUCT_TYPE_TRIGGER_CANDIDATES_PATH = OUTPUT_DIR / 'product_type_trigger_candidates.csv'
PRODUCT_TYPE_REPEATED_TRIGGER_CANDIDATES_PATH = OUTPUT_DIR / 'product_type_trigger_candidates_repeated.csv'
PRODUCT_TYPE_TRIGGER_CANDIDATES_XLSX_PATH = OUTPUT_DIR / 'product_type_trigger_candidates.xlsx'
PRODUCT_TYPE_REPEATED_TRIGGER_CANDIDATES_XLSX_PATH = OUTPUT_DIR / 'product_type_trigger_candidates_repeated.xlsx'

reviewed_product_type_check.to_csv(PRODUCT_TYPE_LITERAL_CHECK_PATH, index=False, encoding='utf-8-sig')
non_literal_cases.to_csv(PRODUCT_TYPE_NON_LITERAL_CASES_PATH, index=False, encoding='utf-8-sig')
non_literal_residual_cases.to_csv(PRODUCT_TYPE_NON_LITERAL_RESIDUAL_CASES_PATH, index=False, encoding='utf-8-sig')
empty_after_subtraction_cases.to_csv(PRODUCT_TYPE_EMPTY_AFTER_SUBTRACTION_PATH, index=False, encoding='utf-8-sig')
product_type_trigger_candidates.to_csv(PRODUCT_TYPE_TRIGGER_CANDIDATES_PATH, index=False, encoding='utf-8-sig')
repeated_trigger_candidates.to_csv(PRODUCT_TYPE_REPEATED_TRIGGER_CANDIDATES_PATH, index=False, encoding='utf-8-sig')
# NEW: для ручной работы удобно сохраняем те же trigger-таблицы ещё и в Excel.
product_type_trigger_candidates.to_excel(PRODUCT_TYPE_TRIGGER_CANDIDATES_XLSX_PATH, index=False)
repeated_trigger_candidates.to_excel(PRODUCT_TYPE_REPEATED_TRIGGER_CANDIDATES_XLSX_PATH, index=False)

print('saved:', PRODUCT_TYPE_LITERAL_CHECK_PATH)
print('saved:', PRODUCT_TYPE_NON_LITERAL_CASES_PATH)
print('saved:', PRODUCT_TYPE_NON_LITERAL_RESIDUAL_CASES_PATH)
print('saved:', PRODUCT_TYPE_EMPTY_AFTER_SUBTRACTION_PATH)
print('saved:', PRODUCT_TYPE_TRIGGER_CANDIDATES_PATH)
print('saved:', PRODUCT_TYPE_REPEATED_TRIGGER_CANDIDATES_PATH)
print('saved:', PRODUCT_TYPE_TRIGGER_CANDIDATES_XLSX_PATH)
print('saved:', PRODUCT_TYPE_REPEATED_TRIGGER_CANDIDATES_XLSX_PATH)


In [ ]:
import pandas as pd

trigger_candidates = pd.read_csv('output/product_type_trigger_candidates.csv')
trigger_candidates.head(30).sort_values('product_type')

## 15. Подключаем отдельный manual trigger-словарь product_type

Теперь ручной словарь лежит в отдельном файле, а не только во вкладке Excel.

Логика шага такая:
- сначала оставляем базовый `product_type_pred`, который уже нашёлся словарями
- потом только для пустых `product_type` пробуем ручной trigger-словарь
- если у правила стоит `category_check_required = yes`, дополнительно проверяем `category_path_keywords`
- сохраняем полный результат, coverage и отдельную таблицу manual-hit кейсов


In [ ]:
import re
import pandas as pd

MANUAL_PRODUCT_TYPE_TRIGGER_DICT_PATH = OUTPUT_DIR / 'product_type_manual_trigger_dictionary_v1.csv'
FIRST_PASS_WITH_MANUAL_PATH = OUTPUT_DIR / 'sklavenitis_brand_product_type_with_manual_triggers.csv'
COVERAGE_WITH_MANUAL_PATH = OUTPUT_DIR / 'brand_product_type_coverage_with_manual_triggers.csv'
MANUAL_TRIGGER_HITS_PATH = OUTPUT_DIR / 'product_type_manual_trigger_hits.csv'
MANUAL_TRIGGER_HITS_XLSX_PATH = OUTPUT_DIR / 'product_type_manual_trigger_hits.xlsx'

manual_trigger_dict = pd.read_csv(MANUAL_PRODUCT_TYPE_TRIGGER_DICT_PATH, encoding='utf-8-sig').fillna('')
first_pass_with_manual = pd.read_csv(FIRST_PASS_PATH, encoding='utf-8-sig').fillna('')

for column in [
    'product_type', 'trigger', 'category_check_required', 'example_full_category_path', 'category_path_keywords', 'notes'
]:
    if column in manual_trigger_dict.columns:
        manual_trigger_dict[column] = manual_trigger_dict[column].astype(str).str.strip()

manual_trigger_dict = manual_trigger_dict.loc[
    manual_trigger_dict['product_type'].ne('') & manual_trigger_dict['trigger'].ne('')
].copy()

def normalize_text_for_manual_triggers(value):
    text = str(value).lower()
    text = re.sub(r'[\u200b\u200c\u200d\ufeff]', '', text)
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    return text

def build_phrase_pattern_for_manual_triggers(term):
    escaped = re.escape(term)
    return re.compile(rf'(?<![a-z0-9]){escaped}(?![a-z0-9])')

def parse_category_keywords(value):
    if not value:
        return []
    return [normalize_text_for_manual_triggers(part) for part in str(value).split('|') if normalize_text_for_manual_triggers(part)]

manual_trigger_dict['product_type'] = manual_trigger_dict['product_type'].map(normalize_text_for_manual_triggers)
manual_trigger_dict['trigger'] = manual_trigger_dict['trigger'].map(normalize_text_for_manual_triggers)
manual_trigger_dict['category_check_required'] = manual_trigger_dict['category_check_required'].map(normalize_text_for_manual_triggers).eq('yes')
manual_trigger_dict['category_keywords_list'] = manual_trigger_dict['category_path_keywords'].map(parse_category_keywords)
manual_trigger_dict['trigger_word_count'] = manual_trigger_dict['trigger'].str.split().str.len()
manual_trigger_dict['trigger_length'] = manual_trigger_dict['trigger'].str.len()
manual_trigger_dict = manual_trigger_dict.sort_values(
    ['trigger_word_count', 'trigger_length', 'category_check_required', 'product_type', 'trigger'],
    ascending=[False, False, False, True, True],
    ignore_index=True,
)

compiled_manual_rules = []
for row in manual_trigger_dict.itertuples(index=False):
    compiled_manual_rules.append({
        'product_type': row.product_type,
        'trigger': row.trigger,
        'pattern': build_phrase_pattern_for_manual_triggers(row.trigger),
        'category_check_required': bool(row.category_check_required),
        'category_keywords_list': list(row.category_keywords_list),
        'notes': row.notes,
    })

def category_matches_for_manual_triggers(category_path, keywords):
    category_norm = normalize_text_for_manual_triggers(category_path)
    if not keywords:
        return False
    return any(keyword in category_norm for keyword in keywords)

final_product_type = []
final_method = []
manual_trigger_used = []
manual_category_keywords_used = []
manual_notes_used = []

for row in first_pass_with_manual.itertuples(index=False):
    existing_pred = normalize_text_for_manual_triggers(row.product_type_pred)
    existing_method = str(row.product_type_match_method).strip()

    if existing_pred:
        final_product_type.append(existing_pred)
        final_method.append(existing_method)
        manual_trigger_used.append('')
        manual_category_keywords_used.append('')
        manual_notes_used.append('')
        continue

    search_text = normalize_text_for_manual_triggers(row.product_core_without_brand or row.product_core)
    category_path = str(row.category_path)

    matched_product_type = ''
    matched_method = ''
    matched_trigger = ''
    matched_keywords = ''
    matched_notes = ''

    for rule in compiled_manual_rules:
        if not rule['pattern'].search(search_text):
            continue

        if rule['category_check_required']:
            if not category_matches_for_manual_triggers(category_path, rule['category_keywords_list']):
                continue
            matched_method = 'product_type_manual_trigger_with_category'
        else:
            matched_method = 'product_type_manual_trigger'

        matched_product_type = rule['product_type']
        matched_trigger = rule['trigger']
        matched_keywords = ' | '.join(rule['category_keywords_list'])
        matched_notes = rule['notes']
        break

    final_product_type.append(matched_product_type)
    final_method.append(matched_method)
    manual_trigger_used.append(matched_trigger)
    manual_category_keywords_used.append(matched_keywords)
    manual_notes_used.append(matched_notes)

first_pass_with_manual['product_type_pred_with_manual'] = final_product_type
first_pass_with_manual['product_type_match_method_with_manual'] = final_method
first_pass_with_manual['manual_trigger_used'] = manual_trigger_used
first_pass_with_manual['manual_category_keywords_used'] = manual_category_keywords_used
first_pass_with_manual['manual_trigger_notes_used'] = manual_notes_used

manual_trigger_hits = first_pass_with_manual.loc[
    first_pass_with_manual['manual_trigger_used'].ne(''),
    [
        'product_core', 'product_core_without_brand', 'category_path', 'product_name',
        'product_type_pred_with_manual', 'product_type_match_method_with_manual',
        'manual_trigger_used', 'manual_category_keywords_used', 'manual_trigger_notes_used'
    ]
].copy()

old_matched_rows = int(first_pass_with_manual['product_type_pred'].astype(str).str.strip().ne('').sum())
new_matched_rows = int(first_pass_with_manual['product_type_pred_with_manual'].astype(str).str.strip().ne('').sum())
manual_added_rows = int(len(manual_trigger_hits))

coverage_with_manual = pd.DataFrame([
    {
        'field': 'product_type_before_manual_triggers',
        'total_rows': len(first_pass_with_manual),
        'matched_rows': old_matched_rows,
        'matched_share': round(old_matched_rows / len(first_pass_with_manual), 4),
        'unknown_rows': int(len(first_pass_with_manual) - old_matched_rows),
    },
    {
        'field': 'product_type_after_manual_triggers',
        'total_rows': len(first_pass_with_manual),
        'matched_rows': new_matched_rows,
        'matched_share': round(new_matched_rows / len(first_pass_with_manual), 4),
        'unknown_rows': int(len(first_pass_with_manual) - new_matched_rows),
        'manual_trigger_added_rows': manual_added_rows,
    },
])

first_pass_with_manual.to_csv(FIRST_PASS_WITH_MANUAL_PATH, index=False, encoding='utf-8-sig')
manual_trigger_hits.to_csv(MANUAL_TRIGGER_HITS_PATH, index=False, encoding='utf-8-sig')
manual_trigger_hits.to_excel(MANUAL_TRIGGER_HITS_XLSX_PATH, index=False)
coverage_with_manual.to_csv(COVERAGE_WITH_MANUAL_PATH, index=False, encoding='utf-8-sig')

print('saved:', MANUAL_PRODUCT_TYPE_TRIGGER_DICT_PATH)
print('saved:', FIRST_PASS_WITH_MANUAL_PATH)
print('saved:', COVERAGE_WITH_MANUAL_PATH)
print('saved:', MANUAL_TRIGGER_HITS_PATH)
print('saved:', MANUAL_TRIGGER_HITS_XLSX_PATH)
print('manual dictionary rows:', len(manual_trigger_dict))
print('manual trigger hit rows:', manual_added_rows)
coverage_with_manual


In [ ]:
manual_trigger_hits.head(20)


## 16. Смотрим unknown product_type после manual trigger-словаря

Этот блок показывает, какие `product_type` всё ещё не нашлись даже после ручного trigger-словаря.

Здесь сохраняем две таблицы:
- полные `unknown`-строки
- сводную top-таблицу по `product_core`, чтобы быстро видеть, что осталось добивать


In [ ]:
UNKNOWN_PRODUCT_TYPE_AFTER_MANUAL_ROWS_PATH = OUTPUT_DIR / 'unknown_product_type_after_manual_triggers_rows.csv'
UNKNOWN_PRODUCT_TYPE_AFTER_MANUAL_PATH = OUTPUT_DIR / 'unknown_product_type_after_manual_triggers_top.csv'
UNKNOWN_PRODUCT_TYPE_AFTER_MANUAL_XLSX_PATH = OUTPUT_DIR / 'unknown_product_type_after_manual_triggers_top.xlsx'

first_pass_with_manual = pd.read_csv(FIRST_PASS_WITH_MANUAL_PATH, encoding='utf-8-sig').fillna('')

unknown_after_manual = first_pass_with_manual.loc[
    first_pass_with_manual['product_type_pred_with_manual'].astype(str).str.strip().eq(''),
    ['product_core', 'product_name', 'category_path', 'product_core_without_brand']
].copy()

unknown_after_manual_top = (
    unknown_after_manual.groupby('product_core', as_index=False)
    .agg(
        count=('product_name', 'size'),
        example_product_name=('product_name', 'first'),
        example_category_path=('category_path', 'first'),
        example_product_core_without_brand=('product_core_without_brand', 'first'),
    )
    .sort_values(['count', 'product_core'], ascending=[False, True], ignore_index=True)
)

unknown_after_manual.to_csv(UNKNOWN_PRODUCT_TYPE_AFTER_MANUAL_ROWS_PATH, index=False, encoding='utf-8-sig')
unknown_after_manual_top.to_csv(UNKNOWN_PRODUCT_TYPE_AFTER_MANUAL_PATH, index=False, encoding='utf-8-sig')
unknown_after_manual_top.to_excel(UNKNOWN_PRODUCT_TYPE_AFTER_MANUAL_XLSX_PATH, index=False)

print('unknown rows after manual triggers:', len(unknown_after_manual))
print('saved:', UNKNOWN_PRODUCT_TYPE_AFTER_MANUAL_ROWS_PATH)
print('saved:', UNKNOWN_PRODUCT_TYPE_AFTER_MANUAL_PATH)
print('saved:', UNKNOWN_PRODUCT_TYPE_AFTER_MANUAL_XLSX_PATH)
unknown_after_manual_top.head(30)


## 17. Итоги coverage по brand и product_type

Для product_type и brand добавила ручные варианты и trigger-слова.

Ниже конечный coverage после всех манипуляций и итерационных доработок в файлах. Считаю итоговые доли покрытия по формуле:

coverage = matched_rows / total_rows

Где:
- total_rows — это общее число строк в полном очищенном датасете Sklavenitis
- matched_rows — это сколько строк алгоритм смог автоматически разметить
- matched_share — это доля таких строк от всего датасета


In [ ]:
COVERAGE_WITH_MANUAL_PATH = OUTPUT_DIR / 'brand_product_type_coverage_with_manual_triggers.csv'

coverage_with_manual = pd.read_csv(COVERAGE_WITH_MANUAL_PATH, encoding='utf-8-sig')
coverage_with_manual


### Как читать эти числа

Для product_type итоговое покрытие после ручного trigger-словаря сейчас высокое, потому что мы уже добавили:
- базовый словарь из reviewed-разметки
- ручные trigger-слова для нестандартных названий
- дополнительные product_type, которые встретились в unknown-остатках
- проверки по категории там, где без нее были бы ложные срабатывания

Для brand покрытие ниже, чем для product_type: бренды чаще бывают редкими, длинными, нестандартно написанными или встречаются только в части ассортимента. Поэтому брендовый слой обычно приходится доразмечать дольше.

Текущие числа уже после применения ручных brand-правил из Excel-файла с unknown brand: из него берутся заполненные brand и trigger_word, после чего эти правила применяются ко всему датасету Sklavenitis.


In [ ]:
def get_coverage_row(field_name):
    return coverage_with_manual.loc[coverage_with_manual['field'] == field_name].iloc[0]

product_type_row = get_coverage_row('product_type_after_manual_triggers')
brand_row = get_coverage_row('brand_after_manual_triggers')

print('Формула: coverage = matched_rows / total_rows')
print('')
print('Product type coverage after manual triggers:')
print(f"{int(product_type_row['matched_rows'])} / {int(product_type_row['total_rows'])} = {product_type_row['matched_share'] * 100:.2f}%")
print('')
print('Brand coverage after manual triggers:')
print(f"{int(brand_row['matched_rows'])} / {int(brand_row['total_rows'])} = {brand_row['matched_share'] * 100:.2f}%")


### Короткая интерпретация результата

product_type сейчас покрыт на 92%, brand сейчас покрыт на 85%, спасибо Господи. Можно переходить к следующему этапу - прогонять на втором супермаркете Alpha Omega


## 18. Alpha Omega transfer

This block reuses the finished Sklavenitis dictionaries and applies the same cleanup idea to `Alpha Omega`.
It saves a cleaned processed file, a first-pass dictionary file, a final file with manual triggers, and a coverage summary in `output/`.


In [ ]:
from scripts.alphaomega_pipeline import run_alphaomega_pipeline

alphaomega_results = run_alphaomega_pipeline()
alphaomega_results['coverage']


In [ ]:
alphaomega_results['final'][[
    'product_name', 'product_core', 'size_raw', 'grade',
    'brand_pred_with_manual', 'product_type_pred_with_manual'
]].head(20)
